# Simple plot of a scan accross the time

### Importing utils

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from pyiconeus.io.base import open_path
from pyiconeus.models.Scan import Scan
from ipywidgets import widgets

Setting up matplotlib for interactive usage

In [ ]:
%matplotlib widget

# Scan loading and setup

### 1) Loading the scan

In [ ]:
scan: Scan = open_path("../tests/data/4Dscan_1_StimVIS16__60_30_60_8_fus3Dv2.source.scan")
voxels = scan.voxels
voxels.shape

### 2) Normalising the values

In [ ]:
norm_voxels = voxels.copy()

In [ ]:
def change_contrast(displayed_voxel, high, low, nTime, pose):
    high /= 100
    low /= 100
    maxVox = np.max(displayed_voxel[:, pose, :, nTime, 0, 0])
    minVox = np.min(displayed_voxel[:, pose, :, nTime, 0, 0])

    high_voxelSpace = high * (maxVox - minVox) + minVox
    low_voxelSpace = low * (maxVox - minVox) + minVox

    displayed_voxel[:, pose, :, nTime, 0, 0] = (displayed_voxel[:, pose, :, nTime, 0, 0] - low_voxelSpace) / (high_voxelSpace - low_voxelSpace)
    displayed_voxel[:, pose, :, nTime, 0, 0] *= 255


### 3) Rotate the image (optional)

In [ ]:
norm_voxels = np.transpose(norm_voxels, axes=(2, 1, 0, 3, 4, 5))

# Interactive display

In [ ]:
with plt.ioff():
    fig1, ax1 = plt.subplots(figsize=(3.5, 3.5))
    fig2, ax2 = plt.subplots(figsize=(3.5, 3.5))
    fig3, ax3 = plt.subplots(figsize=(3.5, 3.5))

for f in (fig1, fig2, fig3):
    f.canvas.header_visible = False
    f.canvas.toolbar_visible = False

im = ax1.imshow(norm_voxels[:, int(scan.sizeY/2), :, int(scan.nTime/2), 0, 0],
                 cmap='gray', interpolation='bilinear', aspect=scan.voxDim.dz/scan.voxDim.dx)
im2 = ax2.imshow(norm_voxels[:, int(scan.sizeY/2), :, int(scan.nTime/2), 0, 0],
                  cmap='gray', interpolation='bilinear', aspect=scan.voxDim.dz/scan.voxDim.dy)
im3 = ax3.imshow(norm_voxels[:, int(scan.sizeY/2), :, int(scan.nTime/2), 0, 0],
                  cmap='gray', interpolation='bilinear', aspect=scan.voxDim.dy/scan.voxDim.dx)

ax1.set_title("Frontal")
ax2.set_title("Sagittal")
ax3.set_title("Axial")

pose_slider  = widgets.IntSlider(value=int(scan.sizeY/2), min=0, max=scan.sizeY-1, description='pose')
xpos_slider  = widgets.IntSlider(value=int(scan.sizeZ/2), min=0, max=scan.sizeZ-1, description='xpos')
zpos_slider  = widgets.IntSlider(value=int(scan.sizeX/2), min=0, max=scan.sizeX-1, description='zpos')
ntime_slider = widgets.IntSlider(value=int(scan.nTime/2), min=0, max=scan.nTime-1, description='nTime')
contrast_high_slider = widgets.FloatSlider(value=100.0, min=0.0, max = 100.0, description='High')
contrast_low_slider = widgets.FloatSlider(value=0.0, min=0.0, max = 100.0, description='Low')

def update(xpos, pose, zpos, nTime, high, low):
    displayed_voxels = norm_voxels.copy()
    change_contrast(displayed_voxels, high, low, nTime, pose)
    
    # Frontal
    img_slice = displayed_voxels[:, pose, :, nTime, 0, 0]
    im.set_data(img_slice)
    im.set_extent([-0.5, img_slice.shape[1]-0.5, img_slice.shape[0]-0.5, -0.5])
    fig1.canvas.draw_idle()

    # Sagittal
    img_slice2 = displayed_voxels[:, :, xpos, nTime, 0, 0]
    im2.set_data(img_slice2)
    im2.set_extent([-0.5, img_slice2.shape[1]-0.5, img_slice2.shape[0]-0.5, -0.5])
    fig2.canvas.draw_idle()

    # Axial
    img_slice3 = displayed_voxels[zpos, :, :, nTime, 0, 0]
    im3.set_data(img_slice3)
    im3.set_extent([-0.5, img_slice3.shape[1]-0.5, img_slice3.shape[0]-0.5, -0.5])
    fig3.canvas.draw_idle()

out = widgets.interactive_output(
    update,
    {'xpos': xpos_slider, 'pose': pose_slider, 'zpos': zpos_slider, 'nTime': ntime_slider, 'high': contrast_high_slider, 'low': contrast_low_slider}
)

col1 = widgets.VBox([fig1.canvas, pose_slider])
col2 = widgets.VBox([fig2.canvas, xpos_slider])
col3 = widgets.VBox([fig3.canvas, zpos_slider])

plots_row = widgets.HBox([col1, col2, col3])
contrast_row = widgets.HBox([contrast_high_slider, contrast_low_slider])

app = widgets.VBox([plots_row, ntime_slider, contrast_row])
display(app)